In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

drive_root = Path("/content/drive/MyDrive")
print("MyDrive exists:", drive_root.exists())

if drive_root.exists():
    print("\nDrive contents:")
    for item in sorted(drive_root.iterdir()):
        print("[DIR]" if item.is_dir() else "[FILE]", item.name)

Mounted at /content/drive
MyDrive exists: True

Drive contents:
[DIR] Colab Notebooks
[FILE] Data_Cleaning_and_Feature_Extraction (2).ipynb
[FILE] EAI-CA3.rar
[FILE] German-20260206T143933Z-1-001.zip
[FILE] Italian-20260206T143958Z-1-001.zip
[FILE] Korean-20260206T144000Z-1-001.zip
[DIR] Quantum 
[FILE] Spanish-20260206T144002Z-1-001.zip
[FILE] TinyMNIST.zip
[DIR] YOLOv8_Fault_Tolerance
[DIR] dataset
[FILE] flowers-20251228T193800Z-3-001.zip
[DIR] flowers_dataset
[FILE] papers .zip


In [2]:
from pathlib import Path

project = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance"
)

print("=== Project folders ===")
for item in sorted(project.iterdir()):
    print("[DIR]" if item.is_dir() else "[FILE]", item.name)

print("\n=== Code files ===")
code = project / "code"

if code.exists():
    for path in sorted(code.rglob("*")):
        if path.is_file() and ".git" not in path.parts:
            print(path.relative_to(project))
else:
    print("code/ folder not found")

=== Project folders ===
[DIR] artifacts
[DIR] code
[DIR] datasets
[DIR] documentation

=== Code files ===
code/.gitignore
code/README.md
code/experiments/baseline/baseline_metadata.json
code/notebooks/00_environment_setup.ipynb
code/notebooks/01_model_baseline.ipynb
code/notebooks/02_baseline_benchmark.ipynb
code/requirements.txt


In [3]:
from pathlib import Path
import sys
import importlib.util

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

print("Project exists:", PROJECT_ROOT.exists())
print("Baseline metadata exists:",
      (PROJECT_ROOT / "experiments/baseline/baseline_metadata.json").exists())
print("Ultralytics installed:",
      importlib.util.find_spec("ultralytics") is not None)
print("Python:", sys.version)

Project exists: True
Baseline metadata exists: True
Ultralytics installed: False
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [4]:
%pip install ultralytics==8.4.189

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [5]:
import ultralytics
import torch
import sys

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.189
PyTorch: 2.11.0+cu130
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
CUDA available: True
GPU: Tesla T4


In [6]:
from pathlib import Path
from urllib.request import urlretrieve
import json
import time
import statistics

import torch
import ultralytics
from ultralytics import YOLO

# مسیر پروژه
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

# محل ذخیره نتایج بنچمارک
RESULTS_DIR = PROJECT_ROOT / "experiments" / "baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# تصویر آزمایش
IMAGE_PATH = Path("/content/bus.jpg")
IMAGE_URL = "https://ultralytics.com/images/bus.jpg"

if not IMAGE_PATH.exists():
    urlretrieve(IMAGE_URL, str(IMAGE_PATH))

# بارگذاری مدل از پیش‌آموزش‌دیده
model = YOLO("yolov8s.pt")

print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Image exists:", IMAGE_PATH.exists())
print("Model loaded: YOLOv8s")

# چند اجرای اولیه برای گرم‌شدن مدل
for _ in range(3):
    model.predict(
        source=str(IMAGE_PATH),
        imgsz=640,
        conf=0.25,
        device=0,
        save=False,
        verbose=False
    )

# اندازه‌گیری چند اجرای مستقل
inference_times_ms = []
wall_times_ms = []
detection_count = None

for i in range(20):
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    results = model.predict(
        source=str(IMAGE_PATH),
        imgsz=640,
        conf=0.25,
        device=0,
        save=False,
        verbose=False
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    wall_ms = (time.perf_counter() - start) * 1000
    wall_times_ms.append(wall_ms)

    # زمان استنتاج گزارش‌شده توسط Ultralytics
    inference_ms = results[0].speed["inference"]
    inference_times_ms.append(float(inference_ms))

    detection_count = len(results[0].boxes)

# خلاصه آماری
benchmark = {
    "model": "YOLOv8s",
    "weights": "yolov8s.pt",
    "ultralytics_version": ultralytics.__version__,
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "image": str(IMAGE_PATH),
    "image_size": 640,
    "confidence_threshold": 0.25,
    "warmup_runs": 3,
    "benchmark_runs": 20,
    "detected_objects": detection_count,
    "inference_mean_ms": statistics.mean(inference_times_ms),
    "inference_std_ms": statistics.stdev(inference_times_ms),
    "inference_median_ms": statistics.median(inference_times_ms),
    "wall_time_mean_ms": statistics.mean(wall_times_ms),
    "wall_time_std_ms": statistics.stdev(wall_times_ms)
}

output_path = RESULTS_DIR / "baseline_benchmark.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(benchmark, f, indent=2, ensure_ascii=False)

print("\n=== BENCHMARK RESULTS ===")
for key, value in benchmark.items():
    print(f"{key}: {value}")

print("\nSaved to:", output_path)

Ultralytics: 8.4.189
GPU: Tesla T4
Image exists: True
Model loaded: YOLOv8s

=== BENCHMARK RESULTS ===
model: YOLOv8s
weights: yolov8s.pt
ultralytics_version: 8.4.189
torch_version: 2.11.0+cu130
gpu: Tesla T4
image: /content/bus.jpg
image_size: 640
confidence_threshold: 0.25
warmup_runs: 3
benchmark_runs: 20
detected_objects: 5
inference_mean_ms: 16.19616190001807
inference_std_ms: 5.426214603223282
inference_median_ms: 13.601545500023349
wall_time_mean_ms: 23.822071749998486
wall_time_std_ms: 5.850176429352975

Saved to: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/baseline/baseline_benchmark.json


In [7]:
import json
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

RESULT_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "baseline"
    / "baseline_benchmark.json"
)

# بررسی اعتبار فایل
with open(RESULT_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

print("JSON valid:", True)
print("Model:", benchmark["model"])
print("Benchmark runs:", benchmark["benchmark_runs"])
print("Mean inference (ms):", round(
    benchmark["inference_mean_ms"], 3
))
print("Result file exists:", RESULT_PATH.exists())

# بررسی وضعیت Git
status = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "status", "--short"],
    capture_output=True,
    text=True,
    check=True
)

print("\n=== Git status ===")
print(status.stdout if status.stdout.strip() else "No changes detected")

JSON valid: True
Model: YOLOv8s
Benchmark runs: 20
Mean inference (ms): 16.196
Result file exists: True

=== Git status ===
 M notebooks/01_model_baseline.ipynb
?? experiments/baseline/baseline_benchmark.json
?? notebooks/02_baseline_benchmark.ipynb



In [8]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

files = [
    "notebooks/01_model_baseline.ipynb",
    "experiments/baseline/baseline_benchmark.json",
    "notebooks/02_baseline_benchmark.ipynb",
]

subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "add", *files],
    check=True
)

subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "status", "--short"],
    check=True
)

CompletedProcess(args=['git', '-C', '/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code', 'status', '--short'], returncode=0)

In [9]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

result = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "status", "--short"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout if result.stdout.strip() else "No changes detected")

A  experiments/baseline/baseline_benchmark.json
M  notebooks/01_model_baseline.ipynb
AM notebooks/02_baseline_benchmark.ipynb

